### Copyright 2022-2026 Crown Copyright

```
Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
```

## Setup Sleeper Client
Set up the logger for the notebook and create a Sleeper Client connected to a Sleeper instance.

Prerequisites:
- Sleeper is already deployed
- A table exists and is partitioned
- AWS credentials and region are configured
- The sleeper Python package is installed

Note: This notebook uses top-level await in code cells. If your environment does not support it, run the calls inside an async function and await that function.

In [ ]:
import logging

from sleeper import SleeperClient, enable_logging

logging.basicConfig(
    level=logging.INFO,
    format="[NOTEBOOK] %(asctime)s %(levelname)s %(message)s",
    force=True,
)

jupyter_logger = logging.getLogger(__name__)

jupyter_logger.info("Starting notebook")

# Enable Sleeper Client logging (switch to DEBUG only when troubleshooting)
enable_logging(logging.INFO)

# TODO: Replace with your deployed values
table_name = "my-table"
instance_id = "instance-123"

sleeper_client = SleeperClient(instance_id)

## Bulk Import Data into Sleeper (Optional)
The next cell bulk imports data into Sleeper from S3 using EMR Serverless.

Skip this if your table already contains suitable data.
If you do run it, make sure the imported data contains keys that you will query in later cells.

In [ ]:
files = ["mybucket/ingest"]

sleeper_client.bulk_import_parquet_files_from_s3(table_name=table_name, files=files, platform="EMRServerless")

jupyter_logger.info("Bulk import request submitted")

## Exact Query
Run an exact query using web sockets via the Sleeper Client.

Update the sample key to a value that exists in your table.

In [ ]:
keys = {"key": ["abcd"]}
exact_results = await sleeper_client.web_socket_exact_key_query(table_name=table_name, keys=keys)

jupyter_logger.info(f"Exact Results: {exact_results}")

## Range Query
Run a range query using web sockets via the Sleeper Client.

Update the sample ranges to match values present in your table.

In [ ]:
keys = [{"value": {"min": "abc", "max": "hij"}}, {"value": {"min": "rst", "max": "xyz"}}]
range_results = await sleeper_client.web_socket_range_key_query(table_name=table_name, keys=keys)

jupyter_logger.info(f"Range Results: {range_results}")